# Stock Trading Agent — Evaluation with LangSmith

This notebook builds a **stock trading assistant** with three tools and evaluates it
using LangSmith datasets and custom scoring.

| Section | What you will learn |
|---------|--------------------|
| **Tools** | `get_price`, `get_news` (Tavily), `buy_ticker` (mock) |
| **Agent** | ReAct agent with `MemorySaver` checkpointer for multi-turn conversations |
| **Evaluation** | Create a LangSmith dataset, define custom evaluators, and run scoring |

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. Required keys in `.env`: `LANGCHAIN_API_KEY`, `TAVILY_API_KEY`, plus the provider key.
4. If packages are missing:
   ```bash
   uv add langsmith yfinance tavily-python langgraph
   ```

In [2]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langsmith import Client
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert LANGCHAIN_API_KEY, "LANGCHAIN_API_KEY not found — add it to your .env file"
assert TAVILY_API_KEY, "TAVILY_API_KEY not found — add it to your .env file"

# ── Enable LangSmith tracing ───────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "stock-trading-eval"

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

ls_client = Client()
logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


---
## Tools

| Tool | Description |
|------|-------------|
| `get_price` | Fetches historical stock prices via Yahoo Finance for a given ticker and date range |
| `get_news` | Searches for related news using Tavily (returns top 3 results) |
| `buy_ticker` | Mock buy-order tool — returns a confirmation string |

In [3]:
@tool
def get_price(ticker: str, date_range: str) -> str:
    """
    """
    logger.info("get_price called | ticker=%s  date_range=%s", ticker, date_range)
    stock = yf.Ticker(ticker)
    hist = stock.history(period=date_range)
    if hist.empty:
        return f"Could not fetch price data for {ticker} over {date_range}."

    latest_close = hist["Close"].iloc[-1]
    period_high = hist["High"].max()
    period_low = hist["Low"].min()
    first_close = hist["Close"].iloc[0]
    change = latest_close - first_close
    pct = (change / first_close) * 100

    result = (
        f"{ticker} ({date_range}): latest close ${latest_close:.2f}, "
        f"high ${period_high:.2f}, low ${period_low:.2f}, "
        f"change {change:+.2f} ({pct:+.2f}%)"
    )
    logger.info("get_price result: %s", result)
    return result


logger.info("Tool ready: get_price")

INFO | Tool ready: get_price


In [4]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def get_news(ticker: str, search_term: str) -> str:
    """
    """
    query = f"{ticker} {search_term}"
    logger.info("get_news called | query='%s'", query)

    response = tavily_client.search(query=query, max_results=3)
    results = response.get("results", [])

    if not results:
        return f"No news found for '{query}'."

    lines = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "No title")
        snippet = r.get("content", "")[:500]
        url = r.get("url", "")
        lines.append(f"{i}. {title}\n   {snippet}\n   {url}")

    result = "\n\n".join(lines)
    logger.info("get_news returned %d results", len(results))
    return result


logger.info("Tool ready: get_news")

INFO | Tool ready: get_news


In [5]:
@tool
def buy_ticker(ticker: str, amount: float, target_price: float) -> str:
    """
    """
    logger.info(
        "buy_ticker called | ticker=%s  amount=%.2f  target_price=%.2f",
        ticker, amount, target_price,
    )
    result = (
        f"Done, a buy order has been set for {ticker} "
        f"with amount of {amount} shares at price ${target_price:.2f}"
    )
    logger.info("buy_ticker result: %s", result)
    return result


logger.info("Tool ready: buy_ticker")

INFO | Tool ready: buy_ticker


---
## Build the Agent

We use `create_react_agent` from LangGraph with a `MemorySaver` checkpointer
so the agent can maintain conversation context across turns.

Each conversation is isolated by `thread_id` in the config.

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = (
    "You are a stock trading assistant."
)

tools = [get_price, get_news, buy_ticker]
checkpointer = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

logger.info("Agent created with tools: %s", [t.name for t in tools])

INFO | Agent created with tools: ['get_price', 'get_news', 'buy_ticker']


---
## Test the Agent

Run a few queries to verify each tool works correctly.
We use the same `thread_id` across turns so the agent remembers context.

In [8]:
config = {"configurable": {"thread_id": "test-session-3"}}

# ── Test 1: Price lookup ───────────────────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="What is the price of NVDA over the this week?")]},
    config=config,
)
print("=== Price Lookup ===")
print(result["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_price called | ticker=NVDA  date_range=1w
ERROR | NVDA: Period '1w' is invalid, must be one of: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Price Lookup ===
I wasn't able to retrieve the price data for NVIDIA (NVDA) over the past week. Would you like me to try again or assist you with something else?


In [9]:
# ── Test 2: News search ────────────────────────────────────────────────────────
result2 = agent.invoke(
    {"messages": [HumanMessage(content="Find me news about Vinamilk related to earnings")]},
    config=config,
)
print("=== News Search ===")
print(result2["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_news called | query='VNM earnings'
INFO | get_news returned 3 results
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== News Search ===
Here are some recent news articles related to Vinamilk (VNM) and its earnings:

1. **Vinamilk (VNM) Earnings Date & Report** - This article provides information about Vinamilk's earnings dates and reports. [Read more here](https://www.investing.com/equities/vietnam-dairy-products-jsc-earnings).

2. **Vietnam Dairy Products Joint Stock Company - Earnings** - This source reports on Vinamilk's earnings per share (EPS) and consensus EPS forecast, along with the latest earnings announcement date. [Read more here](https://www.msn.com/en-us/money/stockdetails/vnm-vn-stock-earnings/fi-aqk1a2?ocid=eldolarnavbar).

3. **About Vietnam Dairy Products JSC (VNM.HM)** - This Reuters article provides real-time stock quotes, news, and financial data about Vinamilk. [Read more here](https://www.reuters.com/markets/companies/VNM.HM/).

If you need more specific information or further assistance, feel free to ask!


In [10]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Should I buy it")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Buy Order ===
To provide a recommendation on whether to buy Vinamilk (VNM), I would need to analyze several factors, including:

1. **Current Price and Historical Performance**: Understanding the current price and how it has performed over time.
2. **Earnings Reports**: Insights from recent earnings reports and forecasts.
3. **Market News**: Any recent news that could impact the stock's performance.
4. **Analyst Ratings**: Recommendations from financial analysts.

Would you like me to gather the current price and any recent earnings reports or news to help make a more informed decision?


In [11]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Yes")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_price called | ticker=VNM  date_range=1w
INFO | get_news called | query='VNM earnings'
INFO | get_news returned 3 results
ERROR | VNM: Period '1w' is invalid, must be one of: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Buy Order ===
I wasn't able to retrieve the current price data for Vinamilk (VNM) over the past week. However, I found some relevant information regarding its earnings:

1. **Vinamilk (VNM) Earnings Date & Report**: The next earnings report is expected on July 28, 2025. The reported earnings were around 1,084.00, with a previous figure of 1,073.42. [Read more here](https://www.investing.com/equities/vietnam-dairy-products-jsc-earnings).

2. **Earnings Report Summary**: The latest earnings announcement was on January 30, 2026, with an EPS of 4028.00 and a consensus EPS forecast of 4086.18. There was an EPS surprise of -1.42%. [Read more here](https://www.msn.com/en-us/money/stockdetails/vnm-vn-stock-earnings/fi-aqk1a2?ocid=eldolarnavbar).

3. **Financial Overview**: Vinamilk's gross profit is reported at 26,209,474.19 million VND, with a net income of 9,410,201.65 million VND for 2025. [Read more here](https://www.reuters.com/markets/companies/VNM.HM/).

### Recommendation Considera

In [12]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Buy 100 shares of it at $370")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | buy_ticker called | ticker=VNM  amount=100.00  target_price=370.00
INFO | buy_ticker result: Done, a buy order has been set for VNM with amount of 100.0 shares at price $370.00
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Buy Order ===
Your buy order for 100 shares of Vinamilk (VNM) at a target price of $370 has been successfully set. If you need any further assistance or have more questions, feel free to ask!


---
## Evaluation with LangSmith

We will:
1. **Create a dataset** with example queries and expected outputs
2. **Define evaluators** — tool-use check + LLM-as-judge for correctness
3. **Run `evaluate()`** and view results in LangSmith

### 1. Create (or load) the Dataset

In [15]:
DATASET_NAME = "stock-trading-agent-eval"

In [ ]:


examples = [
    {
        "inputs": {"question": "What is the price of AAPL over the last month?"},
        "outputs": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain AAPL price data including close price, high, low, and percentage change over the last month.",
        },
    },
    {
        "inputs": {"question": "Find news about NVDA related to AI chips"},
        "outputs": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about NVDA related to AI chips, including titles and links.",
        },
    },
    {
        "inputs": {"question": "Buy 50 shares of TSLA at $250"},
        "outputs": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for TSLA with 50 shares at $250.",
        },
    },
    {
        "inputs": {"question": "What is GOOGL stock price for the past 5 days?"},
        "outputs": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain GOOGL price data over 5 days with close, high, low, and change.",
        },
    },
    {
        "inputs": {"question": "Search for recent AAPL news about iPhone sales"},
        "outputs": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about AAPL and iPhone sales.",
        },
    },
    {
        "inputs": {"question": "Place a buy order for 100 shares of AAPL at $190"},
        "outputs": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for AAPL with 100 shares at $190.",
        },
    },
]

# Delete existing dataset if it exists, then create fresh
existing = list(ls_client.list_datasets(dataset_name=DATASET_NAME))
if existing:
   ls_client.delete_dataset(dataset_id=existing[0].id)
   logger.info("Deleted existing dataset: %s", DATASET_NAME)

dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Evaluation dataset for stock trading agent with price, news, and buy queries.",
)

ls_client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)

logger.info("Dataset '%s' created with %d examples", DATASET_NAME, len(examples))

INFO | Dataset 'stock-trading-agent-eval' created with 6 examples


### 2. Define Evaluators

We define two evaluators:

| Evaluator | What it checks |
|-----------|---------------|
| `correct_tool_used` | Did the agent call the expected tool? (from intermediate messages) |
| `answer_correctness` | LLM-as-judge — does the final answer match the expected criteria? |

In [35]:
import json
from pydantic import BaseModel
def correct_tool_used(outputs: dict, reference_outputs: dict) -> bool:
    """Check whether the agent called the expected tool."""
    expected_tool = reference_outputs.get("expected_tool", "")
    messages = outputs.get("messages", [])
    actual_tools = []
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls: 
                actual_tools.append(tc["name"])
    if expected_tool == "" and actual_tools == []:
        return True
    if expected_tool in actual_tools:
        return True
    logger.debug("Expected tool '%s' was NOT called", expected_tool)
    return False


class AnswerCorrectness(BaseModel):
    answer_correctness: str
    reason: str


judge_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0).with_structured_output(AnswerCorrectness)

async def answer_correctness(outputs: dict, reference_outputs: dict) -> bool:
    """LLM-as-judge: does the final answer satisfy the expected criteria?"""
    actual_answer = outputs.get("messages", [{}])[-1].content
    expected = reference_outputs.get("expected_answer", "")

    instructions = (
        "You are an evaluation judge. Given an actual answer from a stock trading agent "
        "and the expected criteria, determine whether the actual answer satisfies the criteria. "
        "Respond must be in JSON format with the following keys: 'answer_correctness' and 'reason'."
        "The 'answer_correctness' key must be CORRECT or INCORRECT."
        "The 'reason' key must be a string explaining your reasoning for the answer_correctness value."
    )
    user_msg = f"ACTUAL ANSWER:\n{actual_answer}\n\nEXPECTED CRITERIA:\n{expected}"

    response = await judge_llm.ainvoke(
        [
            {"role": "system", "content": instructions},
            {"role": "user", "content": user_msg},
        ]
    )
    
    return {
        "key": "answer_correctness",
        "score": 1 if response.answer_correctness == "CORRECT" else 0,
        "comment": response.reason,
    }


logger.info("Evaluators defined: correct_tool_used, answer_correctness")

INFO | Evaluators defined: correct_tool_used, answer_correctness


### 3. Run Evaluation

We wrap the agent so each evaluation example runs in its own thread (no shared memory).
Then we call `aevaluate()` against the dataset.

In [36]:
from langsmith import aevaluate

eval_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


async def agent_target(inputs: dict) -> dict:
    """Adapter: convert dataset input format to agent invocation."""
    return await eval_agent.ainvoke(
        {"messages": [{"role": "user", "content": inputs["question"]}]},
    )


experiment_results = await aevaluate(
    agent_target,
    data=DATASET_NAME,
    evaluators=[correct_tool_used, answer_correctness],
    experiment_prefix="stock-agent-eval",
    max_concurrency=2,
)

logger.info("Evaluation complete — check results in LangSmith dashboard")

View the evaluation results for experiment: 'stock-agent-eval-7f219e94' at:
https://smith.langchain.com/o/56508c90-17f3-59d2-8160-7e788ed2739a/datasets/2701a68b-bc8f-4152-993c-3b4d94b941e2/compare?selectedSessions=c48fbc2b-cec3-4179-8071-fc5215a56014




0it [00:00, ?it/s]INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
1it [00:03,  3.42s/it]INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_price called | ticker=NVDA  date_range=1w
ERROR | NVDA: Period '1w' is invalid, must be one of: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | buy_ticker called | ticker=AAPL  amount=100.00  target_price=190.00
INFO | buy_ticker result: Done, a buy order has been set for AAPL with amount of 100.0 shares at price $190.00
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2it [00:0

In [34]:
experiment_results.to_pandas()

,inputs.question,outputs.messages,error,reference.expected_answer,feedback.correct_tool_used,feedback.awrapper,execution_time,example_id,id,reference.expected_tool
0,I want to buy shares of Tesla,[content='I want to buy shares of Tesla' addit...,None,Expect to ask about the price and the shares a...,True,None,2.017178,8207a91e-4e66-4126-9483-b55cf6cf82c1,019d53e8-babc-7462-ae60-b949a5de3aa0,NaN
1,What is the price of NVDA over the this week?,[content='What is the price of NVDA over the t...,None,Expect the agent to return the changes in the ...,True,None,3.699326,badb45f1-0fdb-4d4b-8881-2cf10f003943,019d53e8-babd-7763-bb09-77cd2dd0eff8,get_price
2,Place a buy order for 100 shares of AAPL at $190,[content='Place a buy order for 100 shares of ...,None,The response should confirm a buy order for AA...,True,None,2.150230,1c11755c-b628-4e68-add7-9ebd4b2e01db,019d53e8-cb74-75e3-88f9-4a6b3c99b5b5,buy_ticker
3,What is GOOGL stock price for the past 5 days?,[content='What is GOOGL stock price for the pa...,None,The response should contain GOOGL price data o...,True,None,3.298739,2b184437-2c0c-48fd-ad9a-b2f973cce5a2,019d53e8-ce4f-7482-abfa-665adc7d7284,get_price
4,What is the price of AAPL over the last month?,[content='What is the price of AAPL over the l...,None,The response should contain AAPL price data in...,True,None,2.413265,3abff15b-8d1b-4a55-8456-df4b1980ddb2,019d53e8-e15b-72c1-989f-33f12fd3be1c,get_price
5,Buy 50 shares of TSLA at $250,[content='Buy 50 shares of TSLA at $250' addit...,None,The response should confirm a buy order for TS...,True,None,1.876111,486922d7-28f1-43d3-aac6-28a3eda912f4,019d53e8-f076-7013-a4dc-d33dc147054f,buy_ticker
6,Search for recent AAPL news about iPhone sales,[content='Search for recent AAPL news about iP...,None,The response should contain news articles abou...,True,None,7.461749,2f3d7573-bbea-4a22-8307-7b1455b3c025,019d53e8-d8eb-7950-b2ac-5d5bab6810c1,get_news
7,Find news about NVDA related to AI chips,[content='Find news about NVDA related to AI c...,None,The response should contain news articles abou...,True,None,7.325708,bd2def27-9a3c-47f2-9633-72333b2b0dbf,019d53e8-fd92-73b0-8e77-8c89f22eadb2,get_news
